In [218]:
## Imports

import os
import importlib
import random
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-cache")

import numpy as np
import pandas as pd
import hmepy as hm
import hmepy.correlations as hm_correlations
import hmepy.emulator as hm_emulator
import hmepy.emutils as hm_emutils
import hmepy.genpoints as hm_genpoints
import hmepy.modeltraining as hm_modeltraining
from scipy.stats import qmc

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

mipyelq_src = Path("/Users/hamid/mirico/mipyelq/src")
if mipyelq_src.exists() and str(mipyelq_src) not in sys.path:
    sys.path.insert(0, str(mipyelq_src))

for candidate_dir in (Path.cwd(), Path.cwd() / "mirico"):
    if (candidate_dir / "forward_model.py").exists():
        candidate_path = str(candidate_dir.resolve())
        if candidate_path not in sys.path:
            sys.path.insert(0, candidate_path)
        break

if load_dotenv is not None:
    load_dotenv()

from pyelq.source_map import SourceMap
from pyelq.coordinate_system import LLA
from pyelq.gas_species import CH4
from pyelq.dispersion_model.gaussian_plume import GaussianPlume
from pyelq.dispersion_model.turbulence_model import DraxlerModel
from pyelq.sensor.sensor import SensorGroup
from pyelq.sensor.beam import Beam
from pyelq.meteorology import Meteorology
from pyelq.component.background import SpatioTemporalBackground
from pyelq.model import ELQModel
from pyelq.data_access.mirico_data import MiricoData
from pyelq.data_access.xarray import load_from_xarray
from forward_model import ForwardModel

hm_correlations = importlib.reload(hm_correlations)
hm.Correlator = hm_correlations.Correlator
hm_emulator = importlib.reload(hm_emulator)
hm.Emulator = hm_emulator.Emulator
hm_emutils = importlib.reload(hm_emutils)
hm.collectEmulators = hm_emutils.collectEmulators
hm.getRanges = hm_emutils.getRanges
hm_genpoints = importlib.reload(hm_genpoints)
hm.generateNewDesign = hm_genpoints.generateNewDesign
hm_modeltraining = importlib.reload(hm_modeltraining)
hm.emulatorFromData = hm_modeltraining.emulatorFromData

def _to_sensor_compatible(self, n_beam_knots=500, quality_threshold=10):
    if self.settings is None or self.report is None:
        raise ValueError("Missing settings or report data, cannot construct SensorGroup object.")

    sensor_group = SensorGroup()
    valid_report = self.report[self.report["measurement_validity"] >= quality_threshold]

    for identifier in valid_report["pyelq_retro_id"].unique():
        retro_measurements = valid_report[valid_report["pyelq_retro_id"] == identifier]

        sensor = Beam(label=identifier, n_beam_knots=n_beam_knots)
        sensor.time = pd.to_datetime(retro_measurements["timestamp"]).array
        sensor.concentration = retro_measurements["ch4"] * 1e6
        sensor.std_err = retro_measurements["ch4_stderr"] * 1e6

        orion_lla = self.settings[identifier]["orion_lla"]
        retro_lla = self.settings[identifier]["retro_lla"]
        lat_pair, lng_pair, alt_pair = zip(orion_lla, retro_lla)
        sensor.location = LLA(
            latitude=np.array(lat_pair), longitude=np.array(lng_pair), altitude=np.array(alt_pair)
        )

        sensor_group.add_sensor(sensor)

    return sensor_group

def _insert_low_res_wind_compatible(self, meteorology):
    meteorology.time = pd.to_datetime(self.report["timestamp"]).array
    meteorology.u_component = self.report["windy"].to_numpy()
    meteorology.v_component = self.report["windx"].to_numpy()
    meteorology.w_component = self.report["windz"].to_numpy()

def _insert_high_res_wind_compatible(self, meteorology):
    high_res_time_column = "windtime" if "windtime" in self.high_res_wind.columns else "timestamp"
    meteorology.time = pd.to_datetime(self.high_res_wind[high_res_time_column]).array
    meteorology.u_component = self.high_res_wind["windy"].to_numpy()
    meteorology.v_component = self.high_res_wind["windx"].to_numpy()
    meteorology.w_component = self.high_res_wind["windz"].to_numpy()

    for var in ["temperature", "pressure", "wind_turbulence_horizontal", "wind_turbulence_vertical"]:
        setattr(meteorology, var, self._interpolate_low_res_met(getattr(meteorology, var)))

MiricoData.to_sensor = _to_sensor_compatible
MiricoData._insert_low_res_wind = _insert_low_res_wind_compatible
MiricoData._insert_high_res_wind = _insert_high_res_wind_compatible



In [219]:

def _get_coef_model_keep_perfect(data, ranges, outputName, add=False, order=2, verbose=False):
    if verbose:
        print(outputName)

    Xdat = data.loc[:, ranges.keys()]
    Ydat = data.loc[:, outputName]
    if Ydat.nunique(dropna=True) <= 1:
        if verbose:
            print("Output", outputName, "is constant in the training runs. Removing this output from emulation.")
        return None

    if not add and hm_modeltraining.math.comb(len(ranges) + order, len(ranges)) > 2 / 3 * np.shape(data)[0]:
        if verbose:
            print("Maximum number of regression terms is greater than the available degrees of freedom. Changing to add = True.")
        add = True

    direction = "forward" if add else "backward"
    model = hm_modeltraining.Pipeline([
        ("poly", hm_modeltraining.PolynomialFeatures(order, include_bias=False)),
        ("SFS", hm_modeltraining.SequentialFeatureSelector(
            hm_modeltraining.LinearRegression(),
            direction=direction,
            scoring="explained_variance",
            cv=None,
        )),
        ("linear", hm_modeltraining.LinearRegression()),
    ])
    model = model.fit(Xdat, Ydat)

    score = hm_modeltraining.explained_variance_score(model.predict(Xdat), Ydat)
    if score > 0.9999 and verbose:
        print("Regression surface gives a near-perfect fit for", outputName, "- keeping it for this deterministic forward model.")

    return model

hm_modeltraining.getCoefModel = _get_coef_model_keep_perfect
hm.emulatorFromData.__globals__["getCoefModel"] = _get_coef_model_keep_perfect

def _name_to_function_positional(funcName, varNames):
    if funcName == "1":
        return lambda x: 1

    expression = hm_modeltraining.re.sub("\\^(\\d+)", "**\\1", funcName)
    expression = hm_modeltraining.re.sub("\\s", "*", expression)
    for var_name in sorted(varNames, key=len, reverse=True):
        var_index = varNames.index(var_name)
        expression = hm_modeltraining.re.sub(
            rf"(?<![A-Za-z0-9_]){hm_modeltraining.re.escape(var_name)}(?![A-Za-z0-9_])",
            f"x.iloc[{var_index}]",
            expression,
        )
    return eval("lambda x: " + expression)

hm_modeltraining.nameToFunction = _name_to_function_positional
hm.emulatorFromData.__globals__["nameToFunction"] = _name_to_function_positional

## There are some FutureWarnings that are popping up: haven't yet tracked them down so suppressing them
# They might actually be due to incompatibilities with how numpy and pandas behave at present - maybe it gets fixed in updates of those packages?
import warnings
warnings.simplefilter(action = "ignore", category = FutureWarning)

## Loading Data
First, we load the data from the LIVINGLABS zarr store via `pyblobtools`, then convert it to the `MiricoData` object expected by the rest of the notebook.

In [220]:
try:
    import pyblobtools as pbt
except ImportError as exc:
    raise IImportError(
        "pyblobtools is required for the current zarr/server data loader. "
        "Run this notebook with the same Python environment used by "
        "/Users/hamid/mirico/analysis-dev/hamid."
    ) from exc

time_st = "2025-06-18 14:36:00"
time_en = "2025-06-18 15:30:00"
windows = pd.DataFrame(
    [
        {"start": "2025-06-18 14:36:00", "end": "2025-06-18 15:30:00"},
    ]
)
machine = "NG5"
# override_path = "test/datasets/post-modelling/livinglabs/hdf_to_zarr"
extra_loaders = ["hf_wind", "lf_wind", "settings", "environment"]

dataset = pbt.load_zarr(
    machine,
    time_st,
    time_en,
    extra_loaders=extra_loaders,
    # override_path=override_path,
)

liv_data = load_from_xarray(dataset, post_calculate_diffusion_params=True)
liv_data.report["retro_name"] = liv_data.report["retro_name"].apply(
    lambda value: value.decode("utf-8") if isinstance(value, bytes) else value
)
rel_start = str(liv_data.report["timestamp"].min()).split(".", maxsplit=1)[0]
rel_end = str(liv_data.report["timestamp"].max()).split(".", maxsplit=1)[0]
hours = (pd.to_datetime(rel_end) - pd.to_datetime(rel_start)).total_seconds() / 3600

2026-08-11 15:49:52.927 | WARNING  | pyblobtools.load_zarr:_parse_timestamps:24 - Use of string timestamp is ambigious, datetime should be used instead.
2026-08-11 15:49:52.928 | WARNING  | pyblobtools.load_zarr:_parse_timestamps:27 - Use of string timestamp is ambigious, datetime should be used instead.
2026-08-11 15:49:53.147 | DEBUG    | pyblobtools.load_zarr:_get_paths_from_index:64 - Index search returned 1 matching directory.
2026-08-11 15:49:53.341 | DEBUG    | pyblobtools.directory.zarr_directory:locate_between:148 - Searched 1 directory: matched 2 data file(s), 0 icechunk file(s), 1 setup file(s).
2026-08-11 15:49:54.363 | DEBUG    | pyblobtools.readers.zarr_io:_parallel_open:136 - Opened 3/3 files in 1.02s.
2026-08-11 15:49:54.393 | WARNING  | pyblobtools.loaders.base_loader:_timestamp_slice:90 - !! Duplicate timestamps encountered between 2025-06-18T00:14:00.050163968, 2025-06-19T00:20:02.628574976 during zarr ingestion. Ignoring for now...
2026-08-11 15:49:54.631 | DEBUG   

From the above, we extract sensor locations, the retro IDs, and put them together in an array for interrogation later.

In [221]:
sensor_details = liv_data.settings
nsensors = len(np.unique(liv_data.report.retro_id))
sens_vals = []
for v in sensor_details.values():
    sens_vals.append(list(v['retro_lla']))
sens_vals = np.stack(sens_vals)

## Global Configuration Options
We need to define the experiment configurations that we are not interrogating with history matching; that is, the sensor locations as an `LLA` object, which then becomes a `SensorGroup` object. The reference latitude, longitude, and altitude are read from the loaded LIVINGLABS settings.

In [222]:
time_step = "300s"
time_axis = pd.array(
    pd.date_range(start = time_st,
                  end = time_en,
                  freq = time_step),
                  dtype = "datetime64[ns]"
)
nof_obs = time_axis.size
ref_lat, ref_long, ref_alt = next(iter(sensor_details.values()))["orion_lla"]

LLA_obj = LLA()
LLA_obj.from_array(sens_vals)
LLA_arr = LLA_obj.to_array()

nof_sens = len(LLA_arr)
sens_gp = SensorGroup()
for sensor in range(nof_sens):
    new_sensor = Beam()
    new_sensor.label = list(sensor_details.values())[sensor]['retro_name']
    new_sensor.location = LLA(
        latitude = np.array([ref_lat, LLA_obj.latitude[sensor]]),
        longitude = np.array([ref_long, LLA_obj.longitude[sensor]]),
        altitude = np.array([ref_alt, LLA_obj.altitude[sensor]])
    )
    new_sensor.time = time_axis
    new_sensor.concentration = np.zeros(nof_obs)
    sens_gp.add_sensor(new_sensor)

## Observational Data
Next is obtaining the targets to match to, drawn from `report`. Below, the observations are binned into 5-minute intervals: this is controlled by `resample_rate`. Observations are collected by retro, thinned to only include those times for which the retro has a reading, and any results with an `NA` value for `ch4` or `ch4_stderr` are removed.

In [223]:
obs = pd.DataFrame(liv_data.report)
resample_rate = time_step
temp_resamp = obs.set_index("timestamp").ch4.resample(resample_rate).mean()
times = pd.date_range(min(temp_resamp.index), max(temp_resamp.index), freq = resample_rate)
obs_list = list()
for nm in np.unique(obs.retro_name):
    subdat = obs.loc[obs['retro_name'] == nm]
    st_m = subdat.set_index("timestamp").ch4.resample(resample_rate).mean()*1e6
    st_sd = subdat.set_index("timestamp")["ch4_stderr"].resample(resample_rate).mean()*1e6
    st_m.columns = ["timestamp", "mean"]
    st_sd.columns = ["timestamp", "sd"]
    sub_obs_dat = pd.concat([st_m, st_sd], axis = 1)
    which_times = [times[i] in sub_obs_dat.index for i in range(len(times))]
    sub_obs_dat['obs_name'] = [nm + "obs" + str(tim) for tim in pd.Series(range(len(times)))[which_times]]
    obs_list.append(sub_obs_dat)
all_obs = pd.concat(obs_list).dropna()
obs_targets = {all_obs.obs_name.iloc[i]: {'val': all_obs.ch4.iloc[i], 'sigma': all_obs["ch4_stderr"].iloc[i]} for i in range(np.shape(all_obs)[0])}

In [224]:
for i in range(len(times)):
    char1='R4obs'+str(i)
    print(char1+": "+str(obs_targets[char1]))
# obs_targets[["R5obs"+val for val in range(1)]]

R4obs0: {'val': np.float64(2.701003627338406), 'sigma': np.float64(0.008804118973076973)}
R4obs1: {'val': np.float64(5.42064978848411), 'sigma': np.float64(0.010998216628456453)}
R4obs2: {'val': np.float64(6.1827268931013775), 'sigma': np.float64(0.015454509791454057)}
R4obs3: {'val': np.float64(4.0368759744503), 'sigma': np.float64(0.010557279070076298)}
R4obs4: {'val': np.float64(8.22184309926854), 'sigma': np.float64(0.01826518667695243)}
R4obs5: {'val': np.float64(3.1549219406168096), 'sigma': np.float64(0.008172480221807039)}
R4obs6: {'val': np.float64(3.7602569112172635), 'sigma': np.float64(0.01006418612885648)}
R4obs7: {'val': np.float64(6.51543974614535), 'sigma': np.float64(0.012235805409834833)}
R4obs8: {'val': np.float64(5.428220812898737), 'sigma': np.float64(0.0115786810665299)}
R4obs9: {'val': np.float64(5.539032203634253), 'sigma': np.float64(0.011476871724786578)}
R4obs10: {'val': np.float64(5.977400698589164), 'sigma': np.float64(0.011235469111629072)}


## PyELQ Execution
We need a wrapper function for executing `PyELQ` at a given parameter combination: this initialises all other aspects of the model run that depend on parameters. Anything that is a varying parameter (and we therefore include a range for later in `ranges`) gets called here via `pars`.
In this example, the parameters of interest are:
- Wind behaviour via `wind_angle`, `wind_range`, `wind_speed`;
- Turbulence (in principle) via `turb_hor`, `turb_vert`;
- Location of sources via `source1lat`, `source1long`, `source1alt` and similarly for source 2;
- Background concentration via `bg_mean`. Here background is assumed constant, but one could define background as a vector of the same shape as the time axis to incorporate time-varying background behaviour;
- Source concentrations `conc1`, `conc2`.

In [225]:
def run_model(pars):
    ## If using LLA, then uncomment these lines
    #source_locations_LLA = [pars.source1lat, pars.source1long, pars.source1alt]
    #lla = LLA()
    #lla.from_array(np.array([source_locations_LLA]))
    #enus = lla.to_enu(ref_latitude = ref_lat, ref_longitude = ref_long,
    #                   ref_altitude = ref_alt)
    #source_locations = [enus.east[0], enus.north[0], enus.up[0]]
    source_locations = [pars.east1, pars.north1, pars.up1]
    source_emissions = [pars.conc1 * 10**6]
    bg_cont = pars.bg_mean

    model = ForwardModel(data_object = liv_data)
    model.set_meteorology_from_data()
    model.set_sensor_from_data()
    model.gas_species = CH4()
    model.initialise_sources(source_locations, source_emissions)
    df = model.create_simulated_dataframe(background = bg_cont)
    return df
    

## Defining Ranges
The ranges of the parameters to be varied need to be defined, in order to create an initial Latin Hypercube and to specify the bounds of the parameter search. Ranges are not attached to a distribution, and instead correspond to the largest physical extent compatible with our beliefs about the system.

In [226]:
ranges = {
    #"wind_angle": [0, 360],
    #"wind_range": [0, 90],
    #"wind_speed": [0, 10],
    #"turb_hor": [0, 0.2],
    #"turb_vert": [0, 1.5],
    "bg_mean": [1, 3],
    "conc1": [0, 10],
    "east1": [-120, 40],
    "north1": [-10, 120],
    "up1": [-1, 1]
    #"source1lat": [51.57052, 51.57240],
    #"source2lat": [51.57052, 51.57240],
    #"source1long": [-1.316412, -1.314089],
    #"source2long": [-1.316412, -1.314089],
    #"source1alt": [140.0, 141.0],
    #"source2alt": [140.0, 141.0],
    #"conc2": [0, 10]
}

## Running the Initial Ensemble
For the 'zeroth' wave, we create a Latin Hypercube across the space to uniformly fill the region of interest. Typically, we want at least $10$ times as many points as we have varying parameters to train emulators, and some further points to validate: here we have a 50-50 split (so create a LHD of $20$ times the number of parameters).

*An alternative is to create two separate Latin Hypercubes: one for the training set and one for the validation set.*

These points are then passed into the model via the predefined `run_model`, before being reshaped into a `DataFrame` with the appropriate column names and shape.

In [227]:
# Wrapper for running the model on multiple parameter sets, and combining the results into an emulation-digestible result
def normalise_design_points(points):
    if isinstance(points, dict):
        points = points["points"]
    points = pd.DataFrame(points).copy()
    range_names = list(ranges.keys())
    if points.empty and points.shape[1] == 0:
        return pd.DataFrame(columns=range_names)
    if not set(range_names).issubset(points.columns) and points.shape[1] == len(range_names):
        points.columns = range_names
    return points.loc[:, range_names]

def getRuns(points):
    points = normalise_design_points(points)
    if points.empty:
        return pd.DataFrame(columns=list(ranges.keys()))
    output_arr = np.zeros([np.shape(points)[0], np.shape(points)[1]+nof_sens*(nof_obs-1)])
    final_names = None
    for i in np.arange(np.shape(points)[0]):
        res1 = run_model(points.iloc[i,:])
        sens_res = list(points.iloc[i,:])
        sens_nms = list(points.columns)
        for rnm in np.unique(liv_data.report['retro_name']):
            sens_out_vals = res1.set_index(res1.index)["retro_" + rnm].resample(resample_rate).mean()
            sens_res = sens_res + list(sens_out_vals)
            sens_nms = sens_nms + [rnm + "obs" + str(t) for t in np.arange(len(time_axis)-1)]
        final_names = sens_nms
        output_arr[i,:] = sens_res
    return pd.DataFrame(output_arr, columns = final_names)

In [228]:
wave0.shape

(100, 5)

In [229]:
lh_sampler = qmc.LatinHypercube(len(ranges))
unscaled_lh = lh_sampler.random(n = 20*len(ranges))
wave0 = pd.DataFrame(qmc.scale(unscaled_lh,
                               [ran[0] for ran in list(ranges.values())],
                               [ran[1] for ran in list(ranges.values())]),
                               columns = ranges.keys())
# wave0.loc[:,'east1'] = -30
# wave0.loc[:,'north1'] = 50
wave0.loc[:,'conc1'] =wave0.loc[:,'conc1']
wave0_results = getRuns(wave0)

    

ValueError: could not broadcast input array from shape (159,) into shape (145,)

In [ ]:
wave0_results.loc[:,['bg_mean', 'R4obs4','conc1']]
# wave0_results.iloc[0,:]

We thin the data frame to reduce size (the `run_model` function returns sensor readings at every time step, but not all retros have real-world observations).

In [ ]:
has_data = [(nm in list(wave0.columns)) or (nm in list(all_obs['obs_name'])) for nm in list(wave0_results.columns)]
wave0_results = wave0_results.iloc[:,has_data]

## Principal Output Selection
There is little value in emulating every output at every wave, since a comparatively small subset will actually be informative. The `prinVar` function of `hmepy` selects the subset based on the proportion of variance explained: here we seek the set of outputs that explain at least $95\%$ of the variability.

In [ ]:
prinvars = hm.prinVar(wave0_results.iloc[:,len(ranges):(np.shape(wave0_results)[1])], var_cut = 0.95)['ordered_variables']

In [ ]:
prinvars

The model runs obtained are split into the validation and training sets: here done randomly but one could have instead defined two Latin Hypercubes of the same size above, in which case this step is not necessary.

In [ ]:
indices = list(range(np.shape(wave0_results)[0]))
random.shuffle(indices)
train0 = wave0_results.iloc[indices[0:15*len(ranges)],:]
valid0 = wave0_results.iloc[indices[15*len(ranges):np.shape(wave0_results)[0]],:]


In [ ]:
train0.shape

In [ ]:
wave0_results.to_csv("output_debug/wave0.csv")

## Creating Emulators
Finally, we have everything we need to create emulators, using `emulatorFromData`. This takes the training data, the output names we want to emulate, and the ranges of the parameters, and creates an emulator for each output. There are multiple different options in the function: used here is the `verbose = True` option to enable printouts of progress, and the `order = 1` argument to fit no higher than linear regression surfaces (this is justified in early waves, since large-scale behaviour across the space is likely to be well-approximated by a linear regression surface and higher order surfaces are likely to overfit).

Note that, typically, fitting the correlation surface takes a bit of time: this is almost certainly inefficiencies in my code translation from `hmer` (volunteers to speed it up very welcome!)

In [ ]:
## Create emulators
ems = hm.emulatorFromData(train0, prinvars, ranges, verbose = True, order = 2, checkRanges = False)

## Sidenote - Interrogating Emulators
Any of the emulators created can be inspected by printing them: they show the variables chosen in the regression surface, the residual uncertainty (via `u_sigma`) and other aspects like correlation length, chosen correlation function, etc.

In [ ]:
print(ems[2])

In [ ]:
ems[5].getCov(valid0)

## Model Discrepancy and Validation
We do not expect the model to be a perfect representation of reality, and therefore matching the emulators of the model to the actual observations may be problematic if this isn't included. We include some 'external' discrepancy, here chosen to be $2\%$ of the observational value, to account for this.
Finally, we validate the emulators against the (unseen) validation set. Ideally, the output of `validationDiagnostics` should be an empty data frame, but this seldom happens in practice: a handful of points being flagged is not the end of the world.

In [ ]:
# Add some model discrepancy and validate emulators (ideally want an empty DataFrame)
ext_disc_per = 0.02
for em in ems:
    em.disc["external"] = obs_targets[em.outputName]['val']*ext_disc_per
vd = hm.validationDiagnostics(ems, obs_targets, valid0)

In [ ]:
vd

## Stopping Condition 1: Emulator Uncertainty
It is unlikely at the first wave, but one stopping condition for the history matching process is if the emulator uncertainties are smaller than the observational ones (since the emulators are no longer the principal contributor to our uncertainties, and training further emulators is unlikely to be helpful). We can check the ratio of emulator uncertainty to observational uncertainty quite easily.

In [ ]:
em_names = [em.outputName for em in ems]
em_uncerts = [em.uSig for em in ems]
ratios = [em_uncerts[i]/np.sqrt((ext_disc_per * obs_targets[em_names[i]]['val'])**2 + obs_targets[em_names[i]]["sigma"]**2) for i in range(len(em_names))]
ratios

## Generating New Points
Emulators suitably validated, we can generate a collection of *non-implausible* points: parameter combinations that this collection of emulators believes to be suitable for matching to the observational data. This forms the basis for the next set of runs to be put into the model, which in turn will inform more accurate emulators.

In [ ]:
wave1 = hm.generateNewDesign(ems, 20*len(ranges), obs_targets, verbose = True, nth = 1)
wave1 = normalise_design_points(wave1)
wave1

In [ ]:
wave0

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.scatter(wave1['east1'], wave1['north1'], color='blue')
plt.scatter(wave0['east1'], wave0['north1'], color='red')
plt.scatter(wave2['east1'], wave2['north1'], color='green')
plt.xlabel('East 1')
plt.ylabel('North 1')
plt.title('Scatter Plot of East 1 vs North 1')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.scatter(wave1['conc1'], wave1['bg_mean'], color='blue')
plt.scatter(wave0['conc1'], wave0['bg_mean'], color='red')
plt.scatter(wave2['conc1'], wave2['bg_mean'], color='green')
plt.xlabel('Conc 1')
plt.ylabel('Bg Mean')
plt.title('Scatter Plot of Conc 1 vs Bg Mean')
plt.show()

## Subsequent Waves
The process for subsequent waves is very similar to that detailed above, save for a couple of points:
- We set `checkRanges = True` in `emulatorFromData`: this reduces the parameter extent that the emulators are considering to a containing hyperrectangle of the points proposed at the last wave.
- We may change the regression surface order, for example `order = 2` to fit (up to) quadratic surfaces; it is inadvisable to go beyond quadratic order for a variety of reasons but one may do so.
- When proposing new points, we use all emulators trained at each wave: this accounts for the fact that the new emulators are trained on a subspace of the original parameter space and may be emulating different outputs.

The below function collects all of the above steps into a single function, which runs the model for the proposed points, splits the data into a training and validation set, trains the emulators at this wave, and proposes new points. It returns the model runs from this wave, the next design proposed, and the complete set of emulators trained thus far.

In [ ]:
def runWave(prevPoints, targets, nPoints, prevEms = None, order = 1, disc = 0.02):
    prevPoints = normalise_design_points(prevPoints)
    if prevPoints.empty:
        print("No design points available for this wave; stopping.")
        return pd.DataFrame(columns=list(ranges.keys())), prevPoints, prevEms
    print("Running model...")
    newRuns = getRuns(prevPoints)
    print("Selecting variables and creating training and validation set...")
    hasData = [(nm in list(prevPoints.columns)) or (nm in list(targets.keys())) for nm in list(newRuns.columns)]
    newRuns = newRuns.iloc[:,hasData]
    prinvars = hm.prinVar(newRuns.iloc[:,len(ranges):(np.shape(newRuns)[1])], var_cut = 0.95)['ordered_variables']
    indices = list(range(np.shape(newRuns)[0]))
    random.shuffle(indices)
    train = newRuns.iloc[indices[0:15*len(ranges)],:]
    valid = newRuns.iloc[indices[15*len(ranges):np.shape(newRuns)[0]],:]
    print("Training emulators and validating...")
    if (prevEms is None):
        cR = False
    else:
        cR = True
    ems = hm.emulatorFromData(train, prinvars, ranges, order = order, checkRanges = cR)
    for em in ems:
        em.disc['external'] = targets[em.outputName]['val']*disc
    vd = hm.validationDiagnostics(ems, targets, valid)
    if prevEms is None:
        allEms = ems
    else:
        allEms = prevEms + ems
    print("Proposing new design...")
    newDesign = hm.generateNewDesign(allEms, nPoints, targets, opts = {'nth': 1})
    newDesign = normalise_design_points(newDesign)
    return newRuns, newDesign, allEms


In [ ]:
wave1_results, wave2, ems = runWave(wave1, obs_targets, 20*len(ranges), ems)

## Looped Waves
We can of course run this function repeatedly: the below demonstrates collecting the results and looping through waves. Here we have increased the regression surface order to $2$ (quadratic) after the first couple of iterations.

In [ ]:
modelRuns = [wave0_results, wave1_results]
recentPoints = normalise_design_points(wave2)
for i in range(8):
    if recentPoints.empty:
        print("No design points left; stopping wave loop.")
        break
    print("Running wave " + str(i+2) + "...")
    if (i > 1):
        ord = 2
    else:
        ord = 1
    newRuns, newPoints, ems = runWave(recentPoints, obs_targets, 20*len(ranges), ems, order = ord)
    if newRuns.empty:
        break
    modelRuns.append(newRuns)
    recentPoints = normalise_design_points(newPoints)


In [ ]:
len(ems)

## Exporting for Analysis
Both the Python and R packages contain functionality to export and import trained emulators between them. In `hmepy`, the `exportEmulatorToJSON` (`export_emulator_to_json` in `hmer`) exports emulator details into a JSON file which contains all information required to reconstruct the emulators. These files can be imported back into a workflow via `importEmulatorFromJSON` (`import_emulator_from_json`), resulting in the original collection of emulators. This allows, for example, emulators to be trained in a Python workflow and exported to use the visualisation functionality in `hmer`.

*The use of import or export emulator functions requires the most recent commit from the `hmepy` repo; to import into R for analysis it also requires the most recent commit from the `hmer` repo (not the CRAN package).*

These objects contain information about the data used to train them, but not any other model runs (for instance, validation points). It is often useful to save the complete collection of model runs as a csv file: again, this allows use of the plotting functions in `hmer`.

In [ ]:
output_dir = Path("output/")
output_dir.mkdir(parents=True, exist_ok=True)

## While we're here, we may as well run the final ensemble of model runs.
finalPoints = getRuns(recentPoints)
modelRuns.append(finalPoints)

## Export the emulators to a json format
hm.exportEmulatorToJSON(ems, filename = output_dir / "emulators.json")
## Export the observational data
all_obs.to_csv(output_dir / "obs.csv")
## Export each dataframe of model runs to its own csv
for i in range(len(modelRuns)):
    modelRuns[i].to_csv(output_dir / f"wave{i}.csv")


In [ ]:
# Converting source coordinates from ENU to LLA
from pyelq.coordinate_system import ENU

output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

for i, run_df in enumerate(modelRuns):
    converted = run_df.copy()
    coord_cols = ["east1", "north1", "up1"]
    if converted.empty or not set(coord_cols).issubset(converted.columns):
        converted.to_csv(output_dir / f"wave{i}convert.csv")
        continue

    coords = converted.loc[:, coord_cols].apply(pd.to_numeric, errors="coerce")
    valid_coords = coords.notna().all(axis=1)
    if valid_coords.any():
        enu = ENU(ref_latitude=float(ref_lat), ref_longitude=float(ref_long), ref_altitude=float(ref_alt))
        enu_array = coords.loc[valid_coords, coord_cols].astype(float).to_numpy(copy=True)
        enu_array[:, 0:2] *= -1
        enu.from_array(enu_array)
        converted.loc[valid_coords, coord_cols] = enu.to_lla().to_array()

    converted.to_csv(output_dir / f"wave{i}convert.csv")
    modelRuns[i] = converted
